```
<08-3_data_renewal.ipynb>

제미나이 의존도: 50-60%

정확도를 더 높이고 싶어서 여러가지 기술을 추가했다.
LR Scheduler(CosineAnnealingLR)
StratifiedKFold

예전에 캐글 데이터 다룰 때 StratifiedKFold를 써본 적이 있지만, 그때는 거의 제미나이가 코드를 짜주었기 때문에
지금 하는 입장에서 그때의 기억이 어렴풋이 있어서 상당히 어려웠다.
그래서 제미나이 의존도가 높다.

StratifiedKFold의 효과는 대단했다. 정확도는 최고 99.94%까지 올라갔다.
이 가중치를 가진 모델이 틀리는 이미지는 16831개 중에서 14개 뿐이다.
전 모델에서는 바나나도 틀렸는데 얘가 틀린 이미지 중에 바나나는 찾아볼 수 없었다.
이제는 사진의 윤곽도 살펴보는구나 라고 느꼈다.
틀린 이미지를 보면, 대부분 사람들도 틀릴 이미지였다.

다섯 번의 폴드를 통해 얻은 확률값들(outputs.softmax(1))의 평균을 내서 정확도를 구해봤지만 99.94%로 
정확도가 최고였던 fold2의 정확도가 일치했다.
하지만 이것을 모델로 만들기는 불가능하다고 한다.

다음엔.. 가중치 동결 및 해제, 데이터 증강, 조기 종료 의 기술을 써보고 싶다.
```

1. LR Scheduler 추가하기 - CosineAnnealingLR ㅇ
2. 가중치 동결 및 해제(Freeze & Unfreeze)
3. 강력한 데이터 증강 - Albumentations
   RandomBrightnessContrast(조명이 밝았다가 어두워지는 현상)
   GaussNoise(카메라 화질이 안 좋아 자글자글해지는 현상)
   ColorJitter(과일의 색감이 카메라 센서에 따라 조금씩 달라지는 현상)
4. Early Stopping(조기 종료)
5. Train / Val / Test 데이터셋 분리 ㅇ
6. StratifiedKFold 도입 ㅇ

In [17]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

path = "images/kaggle_data/dataset"

fruits = ["apple", "banana", "orange"]
classes = []
data = []

N_SPLITS = 5
BATCH_SIZE = 64


for fruit in fruits:
    path = f"images/kaggle_data/dataset/{fruit}"
    categories = [f"fresh_{fruit}", f"rotten_{fruit}"]
    classes.extend(categories)
    for category in categories:
        image_path = os.path.join(path, category)
        for image in os.listdir(image_path):
            if image.endswith(('jpg', 'jpeg', 'png')):
                data.append({
                    'image_path': os.path.join(image_path, image), 
                    'label_name': category
                })

df = pd.DataFrame(data)
# print(df.head())
# print(classes)

label_map = {name: i for i, name in enumerate(classes)}
df['label'] = df['label_name'].map(label_map)

# Train / Val / Test 나누기 1단계: Test 데이터를 완전히 격리한다.
train_val_df, test_df = train_test_split(
    df, test_size=0.1, stratify=df['label'], random_state=42, shuffle=True
)

# 데이터프레임의 인덱스를 리셋해 주는 것이 안전하다. (쪼개지면서 인덱스가 꼬일 수 있음)
train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Train / Val / Test 나누기 2단계

# 5개의 조각으로 나눠주는 스플릿터 객체를 만든다.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

train_val_df['fold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df['label'])):
    train_val_df.loc[val_idx, 'fold'] = fold_idx

In [11]:
from torch.utils.data import Dataset, DataLoader
import cv2

class FruitsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = row['image_path']
        image = cv2.imread(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image = image)
            image = aug['image']
        
        label = int(row['label'])

        return image, torch.tensor(label, dtype=torch.long)

In [12]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(128, 128), 
    A.HorizontalFlip(p=0.5), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

test_val_transform = A.Compose([
    A.Resize(128, 128), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

In [18]:
import torchvision.models as models
import torch
import torch.nn as nn
import torch.optim as optim

test_dataset = FruitsDataset(df=test_df, transform=test_val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [23]:
from torch.optim.lr_scheduler import CosineAnnealingLR

epochs = 3
kfold = 0

total_test_preds = torch.zeros(len(test_df), len(classes)).to(device)

for kfold in range(5):
    print(f"\n====== FOLD {kfold} START ======")
    train_df = train_val_df[train_val_df['fold'] != kfold].reset_index(drop=True)
    val_df = train_val_df[train_val_df['fold'] == kfold].reset_index(drop=True)
    
    train_dataset = FruitsDataset(df=train_df, transform=train_transform)
    val_dataset = FruitsDataset(df=val_df, transform=test_val_transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 매 폴드마다 '새로운 뇌'로 시작해야 하므로, 모델 초기화를 해줘야 합니다.
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # -----Train Start-----
        model.train()
    
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
    
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
            train_loss += loss.item() * images.size(0)
            
            scheduler.step()
    
            current_lr = scheduler.get_last_lr()[0]
    
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_dataset):.4f} | lr: {current_lr}")
        # -----Train End-----
        
        # -----Validation Start-----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
    
                val_loss += loss.item() * images.size(0)

        val_loss = val_loss / len(val_dataset)
    
        print(f"Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_model_fold{kfold}.pth")
        # -----Validation End-----
        
    # -----Test Start-----
    best_model = models.resnet18(weights=None)
    best_model.fc = nn.Linear(best_model.fc.in_features, len(classes))
    best_model.load_state_dict(torch.load(f"best_model_fold{kfold}.pth"))
    best_model = best_model.to(device)
    best_model.eval()

    test_correct = 0
    
    fold_test_preds = []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = best_model(images.to(device))

            test_correct += (outputs.argmax(1) == labels.to(device)).sum().item()
            
            fold_test_preds.append(outputs.softmax(1))

    print(f"Test Acc: {test_correct/len(test_dataset) * 100:.2f}%")
    
    total_test_preds += torch.cat(fold_test_preds, 0)
    
total_test_preds /= 5


====== FOLD 0 START ======
Epoch 1/3 | Train Loss: 0.1564 | lr: 0.0009046039886902865
Val Loss: 0.1235
Epoch 2/3 | Train Loss: 0.0561 | lr: 0.0006548539886903229
Val Loss: 0.0214
Epoch 3/3 | Train Loss: 0.0744 | lr: 0.0003461460113097342
Val Loss: 0.0257
Test Acc: 98.93%

====== FOLD 1 START ======
Epoch 1/3 | Train Loss: 0.1679 | lr: 0.0009046039886902865
Val Loss: 0.1469
Epoch 2/3 | Train Loss: 0.0771 | lr: 0.0006548539886903229
Val Loss: 0.0363
Epoch 3/3 | Train Loss: 0.0397 | lr: 0.0003461460113097342
Val Loss: 0.0103
Test Acc: 99.47%

====== FOLD 2 START ======
Epoch 1/3 | Train Loss: 0.1510 | lr: 0.0009046039886902865
Val Loss: 0.2633
Epoch 2/3 | Train Loss: 0.0793 | lr: 0.0006548539886903229
Val Loss: 0.0731
Epoch 3/3 | Train Loss: 0.0415 | lr: 0.0003461460113097342
Val Loss: 0.0057
Test Acc: 99.94%

====== FOLD 3 START ======
Epoch 1/3 | Train Loss: 0.1689 | lr: 0.0009046039886902865
Val Loss: 0.2733
Epoch 2/3 | Train Loss: 0.0834 | lr: 0.0006548539886903229
Val Loss: 0.0317
E

In [32]:
final_classes = total_test_preds.argmax(1).cpu().numpy()

actual_labels = torch.tensor([label for _, label in test_dataset]).to(device)
final_ensemble_correct = (total_test_preds.argmax(1) == actual_labels).sum().item()

print(f"Acc: {final_ensemble_correct/len(test_dataset) * 100:.2f}%")

Acc: 99.94%


In [30]:
# 1. 복사본으로 쓸 빈 딕셔너리(그릇)를 하나 준비한다.
# 0번 폴드의 가중치 구조를 그대로 복사해서 가져온다.
weights_0 = torch.load("best_model_fold0.pth")
averaged_weights = {key: torch.zeros_like(val).float() for key, val in weights_0.items()}

# 2. 5개 폴드의 가중치 파일을 읽어와서 모든 레이어의 값을 누적해서 더한다.
for fold in range(5):
    fold_weights = torch.load(f"best_model_fold{fold}.pth")
    for key in averaged_weights.keys():
        averaged_weights[key] += fold_weights[key]

# 3. 5로 나누어 진짜 평균값을 만듭니다.
for key in averaged_weights.keys():
    averaged_weights[key] /= 5

# 4. 완성된 가중치를 단 하나의 파일로 저장한다.
torch.save(averaged_weights, "averaged_model.pth")

In [31]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model.load_state_dict(torch.load("averaged_model.pth"))
model = model.to(device)
model.eval()

correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()

print(f"Acc: {correct/len(test_dataset) * 100:.2f}%")

Acc: 84.26%
